In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType, TimeType,FloatType

In [3]:
from pyspark.sql.functions import col

In [4]:
spark = SparkSession.builder \
    .appName("MiApp") \
    .getOrCreate()    

C:\Users\hever\anaconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [5]:
schema2 = StructType([
    StructField('payment_id',
                StringType(), True),
    StructField('order_id',
                StringType(), True),
    StructField('payment_method',
                StringType(), True),
    StructField('payment_status',
                StringType(), True),
    StructField('amount',
                FloatType(), True),
     StructField('payment_ts',
                DateType(), True),
     StructField('processor',
                StringType(), True),
  
     
    
])

In [6]:
schema3 = StructType([
    StructField('shipment_id',
                StringType(), True),
    StructField('order_id',
                StringType(), True),
    StructField('carrier',
                StringType(), True),
    StructField('tracking_number',
                StringType(), True),
    StructField('shipment_status',
                StringType(), True),
      StructField('shipped_ts',
                DateType(), True),
     StructField('estimated_delivery_ts',
                StringType(), True),
     StructField('delivery_city',
                StringType(), True),
  
     
    
])

In [7]:
ventas = spark.read.csv("../data/batch/shipments.csv", schema=schema3,  header=True)

In [8]:
pagos = spark.read.csv("../data/batch/payments.csv",  schema=schema2,  header=True)

In [9]:
ventas.show()

+---------------+-----------+-------------+---------------+---------------+----------+---------------------+----------------+
|    shipment_id|   order_id|      carrier|tracking_number|shipment_status|shipped_ts|estimated_delivery_ts|   delivery_city|
+---------------+-----------+-------------+---------------+---------------+----------+---------------------+----------------+
|SHP-ORD00000003|ORD00000003|          DHL|  TRK2357166448|      delivered|2025-11-28|  2025-12-01 05:10:51|          Cancún|
|SHP-ORD00000004|ORD00000004|        FedEx|  TRK3736444119|         failed|2026-02-05|  2026-02-09 13:46:22|Ciudad de México|
|SHP-ORD00000006|ORD00000006|        FedEx|  TRK2612639703|      delivered|2026-05-07|  2026-05-09 15:58:54|          Puebla|
|SHP-ORD00000009|ORD00000009|Paquetexpress|  TRK9181026797|      delivered|2025-08-22|  2025-08-25 20:04:33|            León|
|SHP-ORD00000010|ORD00000010|          DHL|  TRK2461909037|        pending|2025-01-07|  2025-01-11 19:18:37|          

In [10]:
pagos.show()

+---------------+-----------+--------------+--------------+---------+----------+-----------+
|     payment_id|   order_id|payment_method|payment_status|   amount|payment_ts|  processor|
+---------------+-----------+--------------+--------------+---------+----------+-----------+
|PAY-ORD00000001|ORD00000001|          cash|      approved| 88700.07|2025-08-11|     stripe|
|PAY-ORD00000002|ORD00000002|        wallet|      approved| 76756.62|2026-03-23|mercadopago|
|PAY-ORD00000003|ORD00000003|        wallet|      approved| 61931.58|2025-11-27|     stripe|
|PAY-ORD00000004|ORD00000004|        wallet|      approved| 48886.77|2026-02-04|mercadopago|
|PAY-ORD00000006|ORD00000006| bank_transfer|      approved|100090.27|2026-05-06|       bank|
|PAY-ORD00000007|ORD00000007|          cash|      approved| 72831.92|2025-09-13|     paypal|
|PAY-ORD00000008|ORD00000008|          cash|      approved|109051.01|2025-01-07|       bank|
|PAY-ORD00000009|ORD00000009| bank_transfer|      approved| 101487.8|2

In [11]:
join1 = pagos.alias("p").join(ventas.alias("v"), col("p.order_id") == col("v.order_id"), "inner")

In [12]:
join1.columns

['payment_id',
 'order_id',
 'payment_method',
 'payment_status',
 'amount',
 'payment_ts',
 'processor',
 'shipment_id',
 'order_id',
 'carrier',
 'tracking_number',
 'shipment_status',
 'shipped_ts',
 'estimated_delivery_ts',
 'delivery_city']

In [13]:
datos_ventas = join1.select("payment_id", "p.order_id", "payment_method", "amount", "payment_ts", "processor", "carrier", "tracking_number", "delivery_city")

In [14]:
datos_ventas.show()

+---------------+-----------+--------------+---------+----------+-----------+-------------+---------------+----------------+
|     payment_id|   order_id|payment_method|   amount|payment_ts|  processor|      carrier|tracking_number|   delivery_city|
+---------------+-----------+--------------+---------+----------+-----------+-------------+---------------+----------------+
|PAY-ORD00000003|ORD00000003|        wallet| 61931.58|2025-11-27|     stripe|          DHL|  TRK2357166448|          Cancún|
|PAY-ORD00000004|ORD00000004|        wallet| 48886.77|2026-02-04|mercadopago|        FedEx|  TRK3736444119|Ciudad de México|
|PAY-ORD00000006|ORD00000006| bank_transfer|100090.27|2026-05-06|       bank|        FedEx|  TRK2612639703|          Puebla|
|PAY-ORD00000009|ORD00000009| bank_transfer| 101487.8|2025-08-21|     stripe|Paquetexpress|  TRK9181026797|            León|
|PAY-ORD00000010|ORD00000010|        wallet| 48346.18|2025-01-05|mercadopago|          DHL|  TRK2461909037|          Cancún|


In [19]:
from pyspark.sql.functions import col, when

# Suponiendo que tu DataFrame original se llama 'df'
df_actualizado = datos_ventas.withColumn(
    "payment_method",
    when(col("payment_method") == "wallet", "cartera_pago")
    .when(col("payment_method") == "bank_transfer", "transferencia_bancaria")
    .when(col("payment_method") == "cash", "dinero")
    .otherwise(col("payment_method")) 
)

df_actualizado.show()

+---------------+-----------+--------------------+---------+----------+-----------+-------------+---------------+----------------+
|     payment_id|   order_id|      payment_method|   amount|payment_ts|  processor|      carrier|tracking_number|   delivery_city|
+---------------+-----------+--------------------+---------+----------+-----------+-------------+---------------+----------------+
|PAY-ORD00000003|ORD00000003|        cartera_pago| 61931.58|2025-11-27|     stripe|          DHL|  TRK2357166448|          Cancún|
|PAY-ORD00000004|ORD00000004|        cartera_pago| 48886.77|2026-02-04|mercadopago|        FedEx|  TRK3736444119|Ciudad de México|
|PAY-ORD00000006|ORD00000006|transferencia_ban...|100090.27|2026-05-06|       bank|        FedEx|  TRK2612639703|          Puebla|
|PAY-ORD00000009|ORD00000009|transferencia_ban...| 101487.8|2025-08-21|     stripe|Paquetexpress|  TRK9181026797|            León|
|PAY-ORD00000010|ORD00000010|        cartera_pago| 48346.18|2025-01-05|mercadopago|

In [1]:
datos_ventas.write.mode("overwrite").parquet("../data/batch/salida_ventas.parquet")

NameError: name 'datos_ventas' is not defined